In [26]:
#Importing the required packages
import glob
import os
from pathlib import Path
from osgeo import gdal
import numpy as np
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import math
import seaborn as sns

import calculate_VICI
import crop_raster
import stack_tif
import kmeans
import cluster_stats
import seasons

print("All packages imported successfully.")

run_clipping = False
run_cropland = False
run_invalid_masking = False
run_clustering = False
run_stats = False

All packages imported successfully.


In [27]:
def flatten(xss):
    return [x for xs in xss for x in xs]
    
def create_zones(cluster_folder,output_dir,invalid_pixel_folder,run_clustering,flag_harmonics):
    output_cube = os.path.join(cluster_folder,"cube_to_cluster.img")
        
    #Create stack of rasters
    stack_folder = os.path.join(output_dir,"stacks")
    os.makedirs(stack_folder,exist_ok=True)
        
    cropped_files = glob.glob(os.path.join(cropped_folder,"*","*"+country+".tif"))
    invalid_pix_files = sorted(glob.glob(os.path.join(invalid_pixel_folder,"*","*"+country+".tif")))[36:-180]
    cropped_stack = os.path.join(stack_folder,"cropped_NDVI_stack.tif")
    invalid_stack = os.path.join(stack_folder,"invalid_pixel_stack.tif")
        
    if run_clustering:
        #scale flag used to scale NDVI from 0-250 to 0-1, not needed for invalid pixels 
        if not os.path.isfile(cropped_stack):
            stack_tif.merge_tifs_to_multiband(cropped_files,cropped_stack)
        if not os.path.isfile(invalid_stack):
            stack_tif.merge_tifs_to_multiband(invalid_pix_files,invalid_stack)
        
        #Create clusters - when rerunning, make sure that the cube is removed:
        if os.path.isfile(output_cube):
            os.remove(output_cube)
        output_CPSZ = r'/home/eoafrica/shared/MOFODRONI/Mali_analysis/Mali_zones.tif'
        #kmeans.cluster_KMeans(cropped_stack,invalid_stack,output_cube,output_CPSZ,num_zones,flag_harmonics)
        print("KMeans clustering finished")    
    
    return output_CPSZ, invalid_stack, stack_folder
    
def define_seasons(output_CPSZ,p50_array,stats_folder,cluster_folder,percentiles,pnum):
    #Define Seasons
        
    with rasterio.open(output_CPSZ) as src:
        cpsz = src.read()
        
    zones = np.unique(cpsz)
        
    seasons_file = os.path.join(stats_folder,"seasons.csv")
        
    seasons_arr = seasons.get_seasons_2(p50_array,cpsz,seasons_file)
        
    seasons_folder = os.path.join(cluster_folder,"season_masks")
    os.makedirs(seasons_folder,exist_ok=True)
        
    seasons.season_mask_2(output_CPSZ,seasons_arr,seasons_folder,percentiles,pnum)

    
def run_VICI_calc(cluster_stats,out_folder,output_dir,bbox,upper_threshold):
    import importlib
    import calculate_VICI
    importlib.reload(calculate_VICI)
    
    VICI_folder = os.path.join(out_folder,"VICI")
    print(f'out vici folder: {VICI_folder}')
    os.makedirs(VICI_folder,exist_ok=True)
    dekads = cluster_stats.get_dekads()
    years = [2023,2024]#[y for y in range(1999,2025)]

    lower_threshold = "p05"

    onlyCropland = True
    processMasks=True

    for year in years:
        for dekad in dekads:
            date = f'{year}{dekad}'

            calculate_VICI.calculate_VICI(date,lower_threshold,upper_threshold,VICI_folder,output_dir,bbox,onlyCropland,processMasks)
    
    

In [28]:
#Determine variables
country = "Mali"
year = 2019

main_dir = Path('/home/eoafrica/shared/MOFODRONI/')

country_shp = os.path.join(main_dir,'Mali_analysis' ,'shapefiles','Mali_buffered.shp')
country_vector = gpd.read_file(country_shp)['geometry']
output_dir = os.path.join(main_dir,"Mali_analysis","Output_new")
os.makedirs(output_dir,exist_ok = True)
savgol_folder = os.path.join(main_dir,"SavGol")
savgol_y = os.path.join(savgol_folder,str(year))
savgol_files = glob.glob(os.path.join(savgol_y,"*.tif"))
cropped_folder = os.path.join(output_dir,"cropped_country")
invalid_pixel_folder = os.path.join(output_dir,"invalid_pixel")
os.makedirs(invalid_pixel_folder,exist_ok=True)


#BoundingBox - Set values to the desired bbox
with rasterio.open('/home/eoafrica/shared/MOFODRONI/Mali_analysis/Mali_zones.tif') as src:
    mali = src.read(1)
    bounds = src.bounds
minX = bounds.left
maxX = bounds.right
minY = bounds.bottom
maxY = bounds.top
bbox = [minX,minY,maxX,maxY]
print(f'Location is {country}, year is {year}')
print(f'bbox is {bbox}')

Location is Mali, year is 2019
bbox is [-12.47767857120003, 9.933035714288216, 4.48660714309672, 25.227678571441025]


In [ ]:
if run_clipping:
    #Cropping the NDVI data to the bbox
    error_list = []
    #Create for-loop here to also loop over years 1999-2024
    for year in range(2000,2020):

        print(f'Processing year {year}')
        savgol_y = os.path.join(savgol_folder,str(year))
        output_y = os.path.join(output_dir,"cropped",str(year))
        country_y = os.path.join(output_dir,"cropped_country",str(year))
        os.makedirs(country_y,exist_ok=True)
        os.makedirs(output_y,exist_ok=True)

        savgol_files = glob.glob(os.path.join(savgol_y,"*.tif"))

        for file in savgol_files:
            try:
                #print(f'Processing {file}')
                input_file = file
                output_file = os.path.join(output_y,Path(file).stem+".tif").replace(".tif",f"_{country}.tif")
                country_file = os.path.join(country_y,Path(file).stem+".tif").replace(".tif",f"_{country}.tif")
                crop_raster.crop(input_file,output_file,bbox)
                with rasterio.open(output_file) as src:
                    out_image, out_transform = rasterio.mask.mask(src, country_vector, crop=True)
                    out_meta = src.meta
                out_meta.update({"driver": "GTiff",
                     "height": out_image.shape[1],
                     "width": out_image.shape[2],
                     "transform": out_transform})
                with rasterio.open(country_file, "w", **out_meta) as dest:
                    dest.write(out_image)

            except Exception as e:
                print(e)
                error_list.append(file)
                continue
        print(f'Year {year} completed successfully.')
        
    # visualise example of results
    print('Example of clipped NDVI upper envelope below:')
    country_open = rasterio.open(country_file,'r').read(1)
    country_scaled = country_open * 0.004
    plt.imshow(country_scaled,cmap='gray')
    plt.title(f'country NDVI upper envelope {country_file.split('_')[-2]}')
    plt.colorbar()
    

In [ ]:
#Creating a masking file for invalid pixels.
run_invalid_masking = True

run_cropland = True

if run_invalid_masking:

    cropped_files = sorted(glob.glob(os.path.join(cropped_folder,"*","*.tif")))
    for cropped_file in cropped_files:
        invalid_file = os.path.join(invalid_pixel_folder,Path(cropped_file).parent.stem,Path(cropped_file).stem+".tif")
        print(invalid_file)
        with rasterio.open(cropped_file) as src:
            data = src.read()
            profile = src.profile

        invalid = data.copy()
        invalid[np.isnan(invalid)]=255
        invalid = invalid.astype("uint8")

        os.makedirs(Path(invalid_file).parent,exist_ok=True)
        profile['nodata']=255
        with rasterio.open(invalid_file, "w", **profile) as dst:
            dst.write(invalid[0,:,:],1)
            
            template_dict = {
            "width":dst.width,
            "height":dst.height,
            "crs":dst.crs,
            "transform":dst.transform,
            "extent":dst.bounds,
            "resolution":dst.res,
            'dtype':profile["dtype"]
            }
        #generate cropland masks
        if run_cropland:
            landcover_folder = os.path.join(output_dir,"landcover")
            os.makedirs(landcover_folder,exist_ok=True)
            
            calculate_VICI.get_landcover_EW(bbox,country_vector,landcover_folder)
            calculate_VICI.get_cropland_mask(landcover_folder,template_dict)
            calculate_VICI.merge_cropmasks(landcover_folder,template_dict)
            run_cropland = False
        
    print("Invalid Pixel Mask is created")

In [30]:
import importlib
import adj_factors
importlib.reload(cluster_stats)
importlib.reload(stack_tif)
importlib.reload(kmeans)
importlib.reload(calculate_VICI)
importlib.reload(adj_factors)
importlib.reload(seasons)

<module 'seasons' from '/home/eoafrica/shared/MOFODRONI/scripts/seasons.py'>

In [31]:
### create zones
u_thresh=15
upper_threshold = "p"+str(u_thresh)
flag_harmonics = False
#Create stack of rasters
stack_folder = os.path.join(output_dir,"stacks")
os.makedirs(stack_folder,exist_ok=True)
out_folder = os.path.join(output_dir,"analysis")

stats_folder = os.path.join(out_folder,"cluster_stats")
os.makedirs(stats_folder,exist_ok=True)
C_Adj_file = os.path.join(stack_folder,'C_stack_adj.tif')
Adjusted_NDVI_file = os.path.join(stack_folder,'Adj_NDVI_stack.tif')

output_cube = os.path.join(output_dir,"cube_to_cluster.img")
        

cropped_files = glob.glob(os.path.join(cropped_folder,"*","*"+country+".tif"))
invalid_pix_files = sorted(glob.glob(os.path.join(invalid_pixel_folder,"*","*"+country+".tif")))
active_dates = [file.split('_')[-2] for file in invalid_pix_files]

cropped_stack = os.path.join(stack_folder,"cropped_NDVI_stack.tif")
invalid_stack = os.path.join(stack_folder,"invalid_pixel_stack.tif")
        
if not os.path.isfile(cropped_stack):
        stack_tif.merge_tifs_to_multiband(cropped_files,cropped_stack)
if not os.path.isfile(invalid_stack):
        stack_tif.merge_tifs_to_multiband(invalid_pix_files,invalid_stack)
        
output_CPSZ = r'/home/eoafrica/shared/MOFODRONI/Mali_analysis/Mali_zones.tif'

print("Zones finished")    

Zones finished


In [6]:
#↓Possible Values: lta, p05, p10, p15, p20, p25, p50, p75, p95, std
stats_list = ["lta","std","p05","p10","p15","p20","p25","p50","p75","p90"]

cluster_stats.cluster_stats(output_CPSZ,invalid_stack,stats_folder,stats_list)

In [17]:
importlib.reload(adj_factors)
adj_factors.compute_adjustments(invalid_stack,output_CPSZ,C_Adj_file,out_folder)

Computing NDVI adjustments per CPS zone...
shape: (720, 1713, 1900)
Dekad 35

In [7]:
importlib.reload(adj_factors)
adj_factors.apply_correction_factor(invalid_stack, C_Adj_file, Adjusted_NDVI_file)

Applying zonal adjustment factor to NDVI values...


In [33]:
importlib.reload(adj_factors)
outdir_percentiles = os.path.join(out_folder,"percentiles")
os.makedirs(os.path.join(out_folder,"percentiles"),exist_ok=True)
percentile_numbers = ['05', '15']
percentiles, p50_array = adj_factors.compute_percentiles(Adjusted_NDVI_file, output_CPSZ, percentile_numbers, outdir_percentiles)

Computing dekadal percentiles per CPS zone...
Dekad 35

In [34]:
with rasterio.open(output_CPSZ,'r') as src:
    cpsz = src.read()
zones=np.unique(cpsz)[1:]

In [35]:
### seasons new version
seasons_file = os.path.join(stats_folder,"seasons.csv")
seasons_arr = adj_factors.get_seasons(p50_array, zones, seasons_file)


Computing seasons...
Finished computing seasons.


In [38]:
importlib.reload(adj_factors)
adj_factors.correct_percentiles_for_seasons(output_CPSZ, seasons_arr,percentiles, percentile_numbers, outdir_percentiles)

Correcting percentiles for seasons and saving them per dekad and CPS zone...
Dekad 35

In [49]:
import importlib
import calculate_VICI
importlib.reload(calculate_VICI)
    
VICI_folder = os.path.join(out_folder,"VICI")
print(f'out vici folder: {VICI_folder}')
os.makedirs(VICI_folder,exist_ok=True)
dekads = cluster_stats.get_dekads()
years = [2019]#[y for y in range(1999,2025)]

lower_threshold,upper_threshold = [f"p{n:02}" for n in percentile_numbers]
onlyCropland = False
processMasks=True

for year in years:
    for dekad in dekads:
        date = f'{year}{dekad}'

        calculate_VICI.calculate_VICI(output_CPSZ,date,lower_threshold,upper_threshold,VICI_folder,output_dir,bbox,active_dates,onlyCropland,processMasks)

out vici folder: /home/eoafrica/shared/MOFODRONI/Mali_analysis/Output_new/analysis/VICI
